In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("urldata.csv")

In [3]:
df

,Unnamed: 0,url,label,result
0,0,https://www.google.com,benign,0
1,1,https://www.youtube.com,benign,0
2,2,https://www.facebook.com,benign,0
3,3,https://www.baidu.com,benign,0
4,4,https://www.wikipedia.org,benign,0
...,...,...,...,...
450171,450171,http://ecct-it.com/docmmmnn/aptgd/index.php,malicious,1
450172,450172,http://faboleena.com/js/infortis/jquery/plugin...,malicious,1
450173,450173,http://faboleena.com/js/infortis/jquery/plugin...,malicious,1
450174,450174,http://atualizapj.com/,malicious,1


In [4]:
df.isnull().sum()

Unnamed: 0    0
url           0
label         0
result        0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.label.value_counts()

label
benign       345738
malicious    104438
Name: count, dtype: int64

In [7]:
from urllib.parse import urlparse
import re

In [14]:
 def extract_domain_and_features(url):
      """
      Extract domain and obfuscation features from URL.
      Returns (domain, features_dict) or (None, {}) if unparseable.
      """
      # Handle non-strings
      if not isinstance(url, str):
          return None, {}

      url = url.strip()
      if not url:
          return None, {}

      # Initialize features
      features = {
          'has_bracket_dot': False,
          'has_paren_dot': False,  # e.g., (dot)
          'has_hxxp': False,
          'obfuscation_score': 0
      }

      # Check for obfuscation patterns BEFORE normalization
      if '[.]' in url:
          features['has_bracket_dot'] = True
          features['obfuscation_score'] += 1

      if '(dot)' in url.lower():
          features['has_paren_dot'] = True
          features['obfuscation_score'] += 1

      if url.lower().startswith('hxxp'):
          features['has_hxxp'] = True
          features['obfuscation_score'] += 1

      # Normalize obfuscation for domain extraction
      url_normalized = url.replace('[.]', '.').replace('(dot)', '.')
      if url_normalized.lower().startswith('hxxp'):
          url_normalized = 'http' + url_normalized[4:]  # replace hxxp with http

      # Add scheme if missing
      if not url_normalized.startswith(('http://', 'https://', 'ftp://','ftps://')):
          url_normalized = 'http://' + url_normalized

      try:
          parsed = urlparse(url_normalized)
          hostname = parsed.hostname

          if hostname is None:
              # Fallback extraction
              netloc = parsed.netloc
              if netloc:
                  if '@' in netloc:
                      netloc = netloc.split('@')[-1]
                  hostname = netloc.split(':')[0]
              else:
                  return None, features

          # Handle IPv6
          if ':' in hostname and hostname.count(':') >= 2:
              # IPv6 address - return as-is (no www to strip)
              return hostname, features

          # Handle regular domain
          if hostname.startswith('www.'):
              hostname = hostname[4:]

          return hostname if hostname else None, features

      except Exception:
          # Last resort regex fallback
          try:
              # Look for domain pattern in normalized URL
              match =re.search(r'[a-zA-Z0-9][a-zA-Z0-9\-]*\.([a-zA-Z]{2,})', url_normalized)
              if match:
                  domain = match.group(0)
                  if domain.startswith('www.'):
                      domain = domain[4:]
                  return domain, features
          except:
              pass
          return None, features

In [15]:
  results = df['url'].apply(lambda x: extract_domain_and_features(x))

In [16]:
  df['domain'] = results.apply(lambda x: x[0])

In [17]:
  features_df = pd.DataFrame(results.apply(lambda x: x[1]).tolist())

In [18]:
  df = pd.concat([df, features_df], axis=1)

In [19]:
df['domain'].notnull().sum()/len(df)

np.float64(0.9999933359397214)

In [20]:
df['has_bracket_dot'].sum()

np.int64(1)

In [21]:
df.head()

,Unnamed: 0,url,label,result,domain,has_bracket_dot,has_paren_dot,has_hxxp,obfuscation_score
0,0,https://www.google.com,benign,0,google.com,False,False,False,0
1,1,https://www.youtube.com,benign,0,youtube.com,False,False,False,0
2,2,https://www.facebook.com,benign,0,facebook.com,False,False,False,0
3,3,https://www.baidu.com,benign,0,baidu.com,False,False,False,0
4,4,https://www.wikipedia.org,benign,0,wikipedia.org,False,False,False,0


In [22]:
features_df.head()

,has_bracket_dot,has_paren_dot,has_hxxp,obfuscation_score
0,False,False,False,0
1,False,False,False,0
2,False,False,False,0
3,False,False,False,0
4,False,False,False,0


In [25]:
results

0         (google.com, {'has_bracket_dot': False, 'has_p...
1         (youtube.com, {'has_bracket_dot': False, 'has_...
2         (facebook.com, {'has_bracket_dot': False, 'has...
3         (baidu.com, {'has_bracket_dot': False, 'has_pa...
4         (wikipedia.org, {'has_bracket_dot': False, 'ha...
                                ...                        
450171    (ecct-it.com, {'has_bracket_dot': False, 'has_...
450172    (faboleena.com, {'has_bracket_dot': False, 'ha...
450173    (faboleena.com, {'has_bracket_dot': False, 'ha...
450174    (atualizapj.com, {'has_bracket_dot': False, 'h...
450175    (writeassociate.com, {'has_bracket_dot': False...
Name: url, Length: 450176, dtype: object